# The Seasons — one `AppSpec` across every agent head

This notebook defines a life-transitions app — change as a cycle of seasons — as a
single **`AppSpec`**, then drives it through the framework's heads:

1. the **standalone Advisor** (machinery hidden, pure counsel),
2. the **Analyst** (transparent Navigator — the user works *inside* the framework),
3. the **Explorer ↔ Advisor counsel toggle** on the same exploration.

Same app, three registers. The spec's pieces map to heads like this:

- `advisor_persona` → standalone Advisor's entire identity
- `voicing` → seasonal flavor layered on the Navigator contract (Analyst/Explorer
  and the counsel toggle)
- `tool_guide` / `tools` → optional; this app has none, showing every field is
  optional (see `minds.ipynb` for an app that brings its own tool)

## Prerequisites

1. **Start Memgraph**: `docker compose -f docker-compose.test.yml up -d`
   (or `/df-memgraph start`).
2. **Configure the LLM**: copy `.env.example` to `.env` and set
   `DIALEXITY_DEFAULT_MODEL` (e.g. `bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0`)
   plus the matching provider credentials.
3. **Run the cells top to bottom.** Jupyter supports top-level `await`.

## 1. Bootstrap

In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.agents.app_spec import AppSpec
from dialectical_framework.agents.advisor.advisor import Advisor
from dialectical_framework.agents.analyst.analyst import Analyst

DialecticalReasoning.setup(Settings.from_env())

case = Case()
case.commit()
print(f"Case ready — sid={case.sid}")

## 2. Declare the app

Two text pieces, no tools. `advisor_persona` is a complete identity (the standalone
Advisor shows nothing else); `voicing` is only a flavor the Navigator heads layer
on their own transparent-framework contract. Neither mentions dialectics or
tools — that's the engine's job.

In [ ]:
SEASONS_APP = AppSpec(
    voicing="""## Seasonal Voicing

Where natural, frame change as seasonal: endings as autumn (release that makes
room), fallow stretches as winter (invisible generative work), beginnings as
spring (growing from what was composted). Locate which season a situation is in —
and which one is being resisted. Use the imagery lightly, in service of clarity.
""",
    advisor_persona="""## Persona

You are a guide for people in transition. You understand change as seasonal: every
ending (autumn) makes room, every fallow stretch (winter) does invisible work, and
every beginning (spring) grows from what was composted before it. You help people
locate which season they are actually in — and which one they are resisting.

When someone is gripping a summer that's ending, you name — kindly but plainly —
the cost of refusing autumn, and the harvest that only release makes possible. When
someone is stuck in a winter and reading it as failure, you reveal the quiet,
generative work that fallow time is doing. You present these as discoveries about
the natural shape of their situation, not as criticism.

When you offer a way forward, you frame it as the next seasonal move — what to let
fall, what to let rest, what to plant — always as an option with real tradeoffs,
never a prescription. You never rush someone toward spring before their winter is
done. Your tone is warm, grounded, and patient, with a long view of time.
""",
)

## 3. Register one: the standalone Advisor

Framework invisible — the persona IS the experience. The Advisor silently builds
the dialectical graph behind every turn.

In [ ]:
with scope(case.sid):
    advisor = Advisor(app=SEASONS_APP)
    reply = await advisor.chat(
        "My kids just left for college and the house is silent. Everyone says "
        "'enjoy the freedom,' but I mostly feel useless, like my main job just "
        "ended and nothing has replaced it."
    )
print(reply)

## 4. Register two: the Analyst (transparent Navigator)

Same spec, different head. The Analyst works *with* the user on the analysis —
tensions, perspectives, and quality scores are all on the table — while the
seasonal `voicing` colors how oppositions are framed. (`advisor_persona` is
ignored here; the Navigator contract stays.)

We use a fresh Case so the two registers don't mix their graphs.

In [ ]:
nav_case = Case()
nav_case.commit()

with scope(nav_case.sid):
    analyst = Analyst(app=SEASONS_APP)
    reply = await analyst.chat(
        "I run a 12-person consultancy. Half my energy says double down on our "
        "classic service line that still pays the bills; the other half says it's "
        "fading and we should retool around AI-assisted delivery. Help me map this "
        "tension properly."
    )
print(reply)

In [ ]:
# Continue working with the Analyst — e.g. develop the tension into perspectives
# and group them for exploration. The Analyst will tell you what it created
# (statements, polarities, perspectives) with their hashes.
with scope(nav_case.sid):
    reply = await analyst.chat(
        "Develop that into full perspectives and, when they're ready, group them "
        "into an exploration so we can look at pathways."
    )
print(reply)

## 5. Register three: the Explorer ↔ Advisor counsel toggle

When the Analyst has created a nexus (the previous cell), the session moves to the
Explorer — and can toggle to counsel mode at any point by handing the SAME
conversation and nexus to an Advisor. Both heads take the SAME `SEASONS_APP`;
the counsel toggle keeps the Navigator contract + seasonal voicing.

Grab the nexus hash from the Analyst's last reply and paste it below.

In [ ]:
from dialectical_framework.agents.explorer.explorer import Explorer

NEXUS_HASH = "..."  # <- paste the nexus hash from the Analyst's reply above

with scope(nav_case.sid):
    explorer = Explorer(
        nexus_hash=NEXUS_HASH, messages=analyst.messages, app=SEASONS_APP
    )
    reply = await explorer.chat("Which causal arrangement looks most plausible?")
print(reply)

In [ ]:
# The toggle: same conversation, same exploration, different register.
# The Advisor pinned to this nexus debriefs what the exploration means —
# still under the Navigator contract, still in seasonal voice.
with scope(nav_case.sid):
    counsel = Advisor(
        nexus_hash=NEXUS_HASH, messages=explorer.messages, app=SEASONS_APP
    )
    reply = await counsel.chat("So what does this all mean for my next quarter?")
print(reply)

# Toggling back is the reverse handover:
#   Explorer(nexus_hash=NEXUS_HASH, messages=counsel.messages, app=SEASONS_APP)

## 6. Try your own

- **Write your own spec.** Pick a domain metaphor, fill `voicing` +
  `advisor_persona`, and rerun sections 3–5 — everything else is unchanged.
- **Add a domain tool** — see `minds.ipynb` for an AppSpec that ships an
  `@llm.tool` + `tool_guide`.
- **Compare the shipped personas**: `from dialectical_framework.agents.apps import
  COUNSELOR_PERSONA, STRATEGIC_ADVISOR_PERSONA` — these are plain preambles for the manual
  layer (`Advisor(app_preamble=...)`); `app=` and `app_preamble=` are mutually
  exclusive.
- **Inspect / resume**: history lives on `agent.messages`; pass it back via
  `messages=` to resume any head.